# Training IndoBERT Sentiment Analysis (5 Classes)

## Objectives
- **Model:** `indobenchmark/indobert-base-p1`
- **Task:** Sentiment Classification
- **Optimization:** GPU (Mixed Precision fp16), Early Stopping, Best Model Loading
- **Target:** High Accuracy without Overfitting

In [ ]:
# 1. Setup & Imports
!pip install transformers datasets scikit-learn accelerate torch pandas seaborn matplotlib

import torch
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Check GPU
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device_name}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# 2. Load Data & Preprocessing
file_path = r'd:/Skripsi/sentiment-analyst-ojol-review/data/gojek_scraped_5class_20251206_130028_FINAL_READY.csv'
df = pd.read_csv(file_path)

# Label Mapping
label_map = {'very_negative': 0, 'negative': 1, 'neutral': 2, 'positive': 3, 'very_positive': 4}
df['label'] = df['sentiment'].map(label_map)

# Split Data (Stratified to keep balance)
# 80% Train, 10% Validation, 10% Test
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['text'].tolist(), 
    df['label'].tolist(), 
    test_size=0.2, 
    stratify=df['label'], 
    random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, 
    temp_labels, 
    test_size=0.5, 
    stratify=temp_labels, 
    random_state=42
)

print(f"Train Size: {len(train_texts)}")
print(f"Val Size:   {len(val_texts)}")
print(f"Test Size:  {len(test_texts)}")

In [ ]:
# 3. Tokenization
model_name = 'indobenchmark/indobert-base-p1'
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_data(texts, labels):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=128)
    dataset = []
    for i in range(len(texts)):
        item = {key: torch.tensor(val[i]) for key, val in encodings.items()}
        item['labels'] = torch.tensor(labels[i])
        dataset.append(item)
    return dataset

train_dataset = tokenize_data(train_texts, train_labels)
val_dataset = tokenize_data(val_texts, val_labels)
test_dataset = tokenize_data(test_texts, test_labels)

In [ ]:
# 4. Metrics Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
# 5. Training Configuration
id2label = {0: 'very_negative', 1: 'negative', 2: 'neutral', 3: 'positive', 4: 'very_positive'}
label2id = {'very_negative': 0, 'negative': 1, 'neutral': 2, 'positive': 3, 'very_positive': 4}

model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=5, 
    id2label=id2label, 
    label2id=label2id
)

# TRAINING ARGUMENTS (Optimized)
training_args = TrainingArguments(
    output_dir='./results_5class',
    num_train_epochs=5,              # Max epochs (Early stopping will likely stop earlier)
    per_device_train_batch_size=16,  # 16 is standard for 6-8GB VRAM
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,   # Virtual batch size = 32
    learning_rate=2e-5,              # Low LR to prevent overfitting
    weight_decay=0.01,               # Regularization
    warmup_ratio=0.1,
    eval_strategy="epoch",         # Check val every epoch
    save_strategy="epoch",           # Save model every epoch
    load_best_model_at_end=True,     # ALWAYS load the best model found
    metric_for_best_model="accuracy",
    fp16=True,                       # GPU Acceleration (Mixed Precision)
    logging_dir='./logs',
    logging_steps=50,
    dataloader_num_workers=0         # Set >0 if on Linux, 0 on Windows usually safer
)

In [ ]:
# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stop if no improve for 2 epochs
)

# START TRAINING
trainer.train()

In [ ]:
# 7. Evaluation on Test Set
print("Evaluating on Test Set...")
test_result = trainer.predict(test_dataset)
print(test_result.metrics)

In [ ]:
# 8. Confusion Matrix & Report
y_preds = np.argmax(test_result.predictions, axis=1)
y_true = test_result.label_ids

print(classification_report(y_true, y_preds, target_names=['very_negative', 'negative', 'neutral', 'positive', 'very_positive']))

# Plot CM
cm = confusion_matrix(y_true, y_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['very_negative', 'negative', 'neutral', 'positive', 'very_positive'], 
            yticklabels=['very_negative', 'negative', 'neutral', 'positive', 'very_positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# 9. Save Final Model
save_path = './saved_model_indobert_5class'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to {save_path}")